#🔥 REAL-WORLD MERGING CHALLENGE (Advanced)
🎯 Scenario (read carefully)

You are a data analyst at a retail company.

You receive 3 messy datasets (simulated from your Superstore data), and your job is to:

Build a clean analytical dataset and extract business insights

#🧩 Step 1: Create Messy Real-World Tables

In [16]:
import pandas as pd
import numpy as np

df = pd.read_csv("/content/sample_data/Superstore sales train.csv")
#df.head(3)

In [17]:
orders = df[[
    "Order ID", "Order Date", "Ship Date", "Ship Mode",
    "Customer ID", "Product ID", "Sales"
]].copy()

In [18]:
orders = orders.sample(frac=1)  # shuffle rows

In [19]:
customers = df[[
    "Customer ID", "Customer Name", "Segment", "Region"
]].copy()

In [20]:
customers = pd.concat([customers, customers.sample(50)])  # duplicates
customers.loc[customers.sample(20).index, "Segment"] = None  # missing values

In [21]:
products = df[[
    "Product ID", "Category", "Sub-Category", "Product Name"
]].copy()

In [22]:
products = products.drop_duplicates()
products.loc[products.sample(20).index, "Category"] = None

**Customer Data Cleaning Strategy**

- Removed duplicate records to ensure uniqueness of customers
- Filled missing Segment values using the most frequent segment per Customer ID
- Remaining missing values were labeled as "Unknown" to avoid incorrect assumptions

This approach preserves data integrity while handling inconsistencies realistically.

In [23]:
customers.isnull().sum()
#customers["Segment"].mode()
#customers["Segment"].value_counts()

,0
Customer ID,0
Customer Name,0
Segment,20
Region,0


In [24]:
customers[customers.duplicated(subset=["Customer ID"]) == True]
customers.drop_duplicates(subset=["Customer ID"], inplace=True)

In [25]:
customers.duplicated(subset=["Customer ID"]).sum()

np.int64(0)

In [26]:
customers.isnull().sum()
customers[customers["Segment"].isna()]

segment_map = (
    customers.groupby("Customer ID")["Segment"]
    .agg(lambda x: x.mode().iat[0] if not x.mode().empty else "Unknown")
)

customers["Segment"] = customers["Customer ID"].map(segment_map)

In [27]:
customers[customers["Segment"].isna()]

,Customer ID,Customer Name,Segment,Region


In [28]:
customers["Segment"].unique()
customers[customers["Segment"] == "Unknown"]

,Customer ID,Customer Name,Segment,Region


**Product Category Validation Strategy**

- Extracted category information from Product ID prefix
- Mapped prefixes (FUR, OFF, TEC) to actual category names
- Used extracted values only to fill missing categories
- Compared extracted vs existing values to validate consistency before applying changes

In [29]:

products.nunique()

,0
Product ID,1861
Category,3
Sub-Category,17
Product Name,1849


In [30]:
products.drop_duplicates(subset=["Product ID"], inplace=True)


In [31]:
products.isnull().sum()
products[products["Category"].isna()]

,Product ID,Category,Sub-Category,Product Name
21,OFF-AR-10000246,None,Art,Newell 318
74,OFF-ST-10004123,None,Storage,Safco Industrial Wire Shelving System
202,OFF-AP-10003622,None,Appliances,"Bravo II Megaboss 12-Amp Hard Body Upright, Re..."
460,TEC-AC-10004571,None,Accessories,Logitech G700s Rechargeable Gaming Mouse
627,OFF-AP-10001563,None,Appliances,Belkin Premiere Surge Master II 8-outlet surge...
766,OFF-BI-10004187,None,Binders,3-ring staple pack
780,OFF-PA-10003657,None,Paper,Xerox 1927
933,OFF-PA-10001745,None,Paper,"Wirebound Message Books, 2 7/8"" x 5"", 3 Forms ..."
985,OFF-EN-10000461,None,Envelopes,"#10- 4 1/8"" x 9 1/2"" Recycled Envelopes"
1279,TEC-AC-10000023,None,Accessories,"Maxell 74 Minute CD-R Spindle, 50/Pack"


In [32]:
products["Category"].unique()

array(['Furniture', 'Office Supplies', 'Technology', None], dtype=object)

In [33]:
products["category-from-id"] = products["Product ID"].str[:3]

products

,Product ID,Category,Sub-Category,Product Name,category-from-id
0,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,FUR
1,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",FUR
2,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,OFF
3,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,FUR
4,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,OFF
...,...,...,...,...,...
9456,TEC-AC-10002380,Technology,Accessories,Sony 8GB Class 10 Micro SDHC R40 Memory Card,TEC
9521,TEC-PH-10002817,Technology,Phones,RCA ViSYS 25425RE1 Corded phone,TEC
9562,TEC-MA-10003589,Technology,Machines,Cisco 8961 IP Phone Charcoal,TEC
9604,OFF-AP-10003099,Office Supplies,Appliances,"Eureka Hand Vacuum, Bagless",OFF


In [34]:
mapping = {
    "FUR": "Furniture",
    "OFF": "Office Supplies",
    "TEC": "Technology"
}


products['category-from-id'] = products['category-from-id'].map(mapping)

In [35]:
products[products["Category"] != products['category-from-id']]

,Product ID,Category,Sub-Category,Product Name,category-from-id
21,OFF-AR-10000246,None,Art,Newell 318,Office Supplies
74,OFF-ST-10004123,None,Storage,Safco Industrial Wire Shelving System,Office Supplies
202,OFF-AP-10003622,None,Appliances,"Bravo II Megaboss 12-Amp Hard Body Upright, Re...",Office Supplies
460,TEC-AC-10004571,None,Accessories,Logitech G700s Rechargeable Gaming Mouse,Technology
627,OFF-AP-10001563,None,Appliances,Belkin Premiere Surge Master II 8-outlet surge...,Office Supplies
766,OFF-BI-10004187,None,Binders,3-ring staple pack,Office Supplies
780,OFF-PA-10003657,None,Paper,Xerox 1927,Office Supplies
933,OFF-PA-10001745,None,Paper,"Wirebound Message Books, 2 7/8"" x 5"", 3 Forms ...",Office Supplies
985,OFF-EN-10000461,None,Envelopes,"#10- 4 1/8"" x 9 1/2"" Recycled Envelopes",Office Supplies
1279,TEC-AC-10000023,None,Accessories,"Maxell 74 Minute CD-R Spindle, 50/Pack",Technology


In [36]:
products["Category"] = products["Category"].fillna(products["category-from-id"])

In [37]:
products.drop(columns="category-from-id", inplace=True)

**Orders Validation**

In [38]:
orders
orders.isnull().sum()
orders[orders.duplicated() == True]
orders.drop_duplicates(inplace=True)

# PART 2: Merge Strategically

In [39]:
customers.columns
products.columns
orders.columns

Index(['Order ID', 'Order Date', 'Ship Date', 'Ship Mode', 'Customer ID',
       'Product ID', 'Sales'],
      dtype='object')

In [40]:
merged_1 = pd.merge(orders, customers, on="Customer ID", how="outer")
merged_1
merged_1.isnull().sum()

,0
Order ID,0
Order Date,0
Ship Date,0
Ship Mode,0
Customer ID,0
Product ID,0
Sales,0
Customer Name,0
Segment,0
Region,0


In [41]:
final_df = pd.merge(merged_1, products, on="Product ID", how="outer")
final_df

,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Product ID,Sales,Customer Name,Segment,Region,Category,Sub-Category,Product Name
0,CA-2018-140326,04/09/2018,06/09/2018,First Class,HW-14935,FUR-BO-10000112,825.174,Helen Wasserman,Corporate,West,Furniture,Bookcases,"Bush Birmingham Collection Bookcase, Dark Cherry"
1,CA-2016-130785,05/09/2016,09/09/2016,Standard Class,AG-10900,FUR-BO-10000330,411.332,Arthur Gainer,Consumer,West,Furniture,Bookcases,"Sauder Camden County Barrister Bookcase, Plank..."
2,CA-2018-125472,30/05/2018,31/05/2018,First Class,BD-11725,FUR-BO-10000330,241.960,Bruce Degenhardt,Consumer,South,Furniture,Bookcases,"Sauder Camden County Barrister Bookcase, Plank..."
3,CA-2015-105249,28/11/2015,28/11/2015,Same Day,DH-13675,FUR-BO-10000330,411.332,Duane Huffman,Home Office,East,Furniture,Bookcases,"Sauder Camden County Barrister Bookcase, Plank..."
4,CA-2016-118423,24/03/2016,27/03/2016,First Class,DP-13390,FUR-BO-10000362,359.058,Dennis Pardue,Home Office,Central,Furniture,Bookcases,Sauder Inglewood Library Bookcases
...,...,...,...,...,...,...,...,...,...,...,...,...,...
9794,CA-2017-117590,08/12/2017,10/12/2017,First Class,GH-14485,TEC-PH-10004977,1097.544,Gene Hale,Corporate,Central,Technology,Phones,GE 30524EE4
9795,CA-2018-167395,02/12/2018,04/12/2018,First Class,KM-16720,TEC-PH-10004977,979.950,Kunst Miller,Consumer,West,Technology,Phones,GE 30524EE4
9796,CA-2018-144498,06/05/2018,06/05/2018,Same Day,MB-18085,TEC-PH-10004977,627.168,Mick Brown,Consumer,East,Technology,Phones,GE 30524EE4
9797,CA-2015-167199,06/01/2015,10/01/2015,Standard Class,ME-17320,TEC-PH-10004977,391.980,Maria Etezadi,Home Office,South,Technology,Phones,GE 30524EE4


In [42]:
final_df.shape
final_df.isnull().sum()

,0
Order ID,0
Order Date,0
Ship Date,0
Ship Mode,0
Customer ID,0
Product ID,0
Sales,0
Customer Name,0
Segment,0
Region,0


# **Buisness Insights**


**Customers with most revenue**

In [43]:
final_df.groupby("Customer Name").sum("Sales").sort_values(by="Sales", ascending=False).head(3)

,Sales
Customer Name,
Sean Miller,25043.050
Tamara Chand,19052.218
Raymond Buch,15117.339


**Top 5 customers by each region**


In [44]:
top_by_region = (
    df.groupby(["Region", "Customer ID", "Customer Name"])["Sales"]
    .sum()
    .reset_index()
)
top_by_region

top_by_region["Rank"] = (
    top_by_region
    .groupby("Region")["Sales"]
    .rank(method="dense", ascending=False)
)
top_by_region

top_by_region[top_by_region["Rank"] <= 5].sort_values(by=["Region", "Rank"])

,Region,Customer ID,Customer Name,Sales,Rank
585,Central,TC-20980,Tamara Chand,18437.1380,1.0
6,Central,AB-10105,Adrian Barton,12181.5940,2.0
68,Central,BM-11140,Becky Martin,10539.8960,3.0
524,Central,SC-20095,Sanjit Chand,9900.1900,4.0
260,Central,HM-14860,Harry Marie,6621.4788,5.0
1242,East,TA-21385,Tom Ashbrook,13723.4980,1.0
904,East,HL-15040,Hunter Lopez,10522.5500,2.0
715,East,BS-11365,Bill Shonely,10022.2930,3.0
891,East,GT-14710,Greg Tran,9382.9340,4.0
1233,East,SV-20365,Seth Vernon,9216.5680,5.0


**Which category dominates each region?**

In [48]:
region_category = (
    df.groupby(["Region", "Category"])["Sales"]
    .sum()
    .reset_index()
)

top_category = region_category.loc[
    region_category.groupby("Region")["Sales"].idxmax()
]

top_category

,Region,Category,Sales
2,Central,Technology,168739.208
5,East,Technology,263116.527
8,South,Technology,148195.208
11,West,Technology,247404.930


**How did missing values affect your analysis?**

Missing values primarily affected categorical fields such as customer segment and product category. In the customers table, missing segment values required imputation using the most frequent segment per customer, which may introduce bias by assuming consistent customer behavior.

In the products table, missing category values were reconstructed using encoded information from the Product ID, which improved completeness but relied on the correctness of the encoding structure.

Additionally, missing values in key columns could have caused incomplete joins, leading to potential data loss or inaccuracies in aggregated metrics such as total sales.

**Which table caused the most issues?**

The customers table posed the most challenges due to missing segment values, duplicate customer IDs, and inconsistencies within customer records. These issues directly impacted grouping, segmentation, and the integrity of the merged dataset.

In comparison, the products table required moderate cleaning, mainly involving category reconstruction, while the orders table was relatively clean and structured.

**Hidden Problem Detection**         
Are there orders with missing customer info after merge?         
Are there products without category?

In [47]:
final_df[final_df["Customer Name"].isna()]
final_df[final_df["Category"].isna()]

,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Product ID,Sales,Customer Name,Segment,Region,Category,Sub-Category,Product Name
